# Oil & Gas: Africa–USA Crude Trade & Production Analysis

This notebook explores real **World Bank** indicators for 11 major African oil
producers and the USA. We examine oil dependency, energy use, FDI inflows,
merchandise exports, and the economic footprint of petroleum.

**Data:** `data/processed/oil_gas_master.csv` (produced by the ETL pipeline).


In [ ]:
import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


In [ ]:
# Load the processed master table
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
csv_path = PROJECT_ROOT / "data" / "processed" / "oil_gas_master.csv"
df = pd.read_csv(csv_path)
df.head()


## Exploratory Data Analysis

In [ ]:
print("Shape:", df.shape)
print("\nColumns & dtypes:")
print(df.dtypes)
print("\nYear range:", df["year"].min(), "-", df["year"].max())
print("Countries:", df["iso3"].nunique())


In [ ]:
# Summary statistics
df.describe()


In [ ]:
# Missing values per column
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]


### Visualization 1 — Top 10 African countries by average oil rents (% GDP)

In [ ]:
afr = df[df["is_african"] == 1]
top = (afr.groupby("country_name")["oil_rents_pct_gdp"]
          .mean().sort_values(ascending=False).head(10))
plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index, palette="rocket")
plt.xlabel("Average oil rents (% of GDP)")
plt.ylabel("")
plt.title("Top 10 African Countries by Average Oil Rents (% of GDP)")
plt.tight_layout()
plt.show()


### Visualization 2 — Oil rents trend over years

In [ ]:
focus = ["Nigeria", "Angola", "Libya", "Algeria", "United States"]
sub = df[df["country_name"].isin(focus)]
plt.figure(figsize=(11, 6))
sns.lineplot(data=sub, x="year", y="oil_rents_pct_gdp",
             hue="country_name", marker="o")
plt.xlabel("Year")
plt.ylabel("Oil rents (% of GDP)")
plt.title("Oil Rents Trend: Nigeria, Angola, Libya, Algeria vs USA")
plt.legend(title="Country")
plt.tight_layout()
plt.show()


### Visualization 3 — GDP per capita vs oil rents (Plotly)

In [ ]:
scatter_df = df.dropna(subset=["gdp_per_capita_usd", "oil_rents_pct_gdp",
                                "merchandise_exports_usd"]).copy()
scatter_df["region"] = np.where(scatter_df["is_african"] == 1, "Africa", "USA")
fig = px.scatter(
    scatter_df,
    x="gdp_per_capita_usd",
    y="oil_rents_pct_gdp",
    color="region",
    size="merchandise_exports_usd",
    hover_name="country_name",
    hover_data=["year"],
    title="GDP per Capita vs Oil Rents (bubble size = merchandise exports)",
    labels={"gdp_per_capita_usd": "GDP per capita (USD)",
            "oil_rents_pct_gdp": "Oil rents (% of GDP)"},
)
fig.show()


### Visualization 4 — Correlation matrix of numeric indicators

In [ ]:
num_cols = ["oil_rents_pct_gdp", "natural_resources_rents_pct_gdp",
            "gdp_per_capita_usd", "energy_use_kg_oil_eq",
            "electricity_from_oil_pct", "merchandise_exports_usd",
            "fdi_inflows_usd", "oil_dependency_ratio"]
corr = df[num_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix of Oil & Gas Indicators")
plt.tight_layout()
plt.show()


### Visualization 5 — FDI inflows: African producers vs USA (latest year)

In [ ]:
fdi = df.dropna(subset=["fdi_inflows_usd"])
latest_year = int(fdi["year"].max())
latest = fdi[fdi["year"] == latest_year].copy()
latest["fdi_bn_usd"] = latest["fdi_inflows_usd"] / 1e9
latest = latest.sort_values("fdi_bn_usd", ascending=False)
fig = px.bar(
    latest,
    x="country_name",
    y="fdi_bn_usd",
    color=np.where(latest["is_african"] == 1, "Africa", "USA"),
    title=f"FDI Net Inflows by Country ({latest_year})",
    labels={"country_name": "Country", "fdi_bn_usd": "FDI inflows (billion USD)",
            "color": "Region"},
)
fig.show()


## Key Insights

- **Concentrated oil dependency:** A handful of African producers (Libya, Congo,
  Angola, Equatorial Guinea, Gabon) draw a very large share of GDP from oil
  rents, while the USA's share is negligible.
- **Resource curse signal:** Countries with high oil rents frequently show low
  GDP per capita, hinting at limited economic diversification.
- **Volatility:** Oil-rent trends swing sharply with global crude prices,
  exposing dependent economies to boom-and-bust cycles.
- **Energy divide:** Per-capita energy use in the USA dwarfs the African
  average, despite Africa being a major crude supplier.
- **Capital & trade concentration:** FDI inflows and merchandise exports cluster
  in the largest economies (Nigeria, Angola, Egypt, USA), showing how petroleum
  steers investment and trade flows.
